In [1]:
import os
import requests
import pandas as pd
import time

In [2]:

# 1. Definición de las ciudades que mueven la aguja energética en Texas
ciudades = {
    "Houston": {"lat": 29.7604, "lon": -95.3698},
    "Dallas": {"lat": 32.7767, "lon": -96.7970},
    "Austin": {"lat": 30.2672, "lon": -97.7431}
}

# Rango temporal idéntico al de la demanda
START_DATE = "2022-01-01"
END_DATE = "2025-12-31"

ENDPOINT_URL = "https://archive-api.open-meteo.com/v1/archive"

dataframes_clima = []

In [3]:

print(f"Iniciando descarga de datos climáticos desde {START_DATE} hasta {END_DATE}...\n")

for nombre_ciudad, coords in ciudades.items():
    print(f"⏳ Extrayendo datos para {nombre_ciudad}...")
    
    # Parámetros para Open-Meteo Archive
    # Pedimos: temperatura a 2m, humedad relativa, temperatura aparente (sensación térmica) y velocidad del viento
    params = {
        "latitude": coords["lat"],
        "longitude": coords["lon"],
        "start_date": START_DATE,
        "end_date": END_DATE,
        "hourly": "temperature_2m,relative_humidity_2m,apparent_temperature,wind_speed_10m",
        "timezone": "UTC"  # Lo pedimos en UTC para que machee directo con el 'period' de la EIA
    }
    
    try:
        response = requests.get(ENDPOINT_URL, params=params)
        response.raise_for_status()
        res_json = response.json()
        
        # Parsear los datos horarios
        hourly_data = res_json["hourly"]
        
        # Crear el DataFrame para esta ciudad específica
        df_ciudad = pd.DataFrame({
            "timestamp_utc": hourly_data["time"],
            f"{nombre_ciudad}_temp": hourly_data["temperature_2m"],
            f"{nombre_ciudad}_humidity": hourly_data["relative_humidity_2m"],
            f"{nombre_ciudad}_apparent_temp": hourly_data["apparent_temperature"],
            f"{nombre_ciudad}_wind_speed": hourly_data["wind_speed_10m"]
        })
        
        # Guardar en nuestra lista de dataframes
        dataframes_clima.append(df_ciudad)
        time.sleep(1) # Pausa de cortesía
        
    except Exception as e:
        print(f"❌ Error al descargar datos de {nombre_ciudad}: {e}")

# 2. Consolidación de los datos climáticos
if len(dataframes_clima) == 3:
    print("\nConsolidando matrices climáticas...")
    # Hacemos un merge de las 3 ciudades usando la columna temporal como llave
    df_clima_completo = dataframes_clima[0]
    df_clima_completo = pd.merge(df_clima_completo, dataframes_clima[1], on="timestamp_utc")
    df_clima_completo = pd.merge(df_clima_completo, dataframes_clima[2], on="timestamp_utc")
    
    # Agregar una columna promedio del estado (feature engineering preliminar)
    df_clima_completo["texas_avg_temp"] = df_clima_completo[["Houston_temp", "Dallas_temp", "Austin_temp"]].mean(axis=1)
    
    print(f"¡Éxito! Dataset climático generado con {df_clima_completo.shape[0]} registros horarios.")
    print(df_clima_completo[["timestamp_utc", "texas_avg_temp", "Houston_temp", "Dallas_temp"]].head())
    
    # Guardar localmente de forma estática
    os.makedirs('data', exist_ok=True)
    df_clima_completo.to_csv('data/texas_weather_2022_2025_static.csv', index=False)
    print("\n[OK] Datos climáticos respaldados en: data/texas_weather_2022_2025_static.csv")
else:
    print("Error: No se pudieron recolectar los datos de las tres ciudades.")

Iniciando descarga de datos climáticos desde 2022-01-01 hasta 2025-12-31...

⏳ Extrayendo datos para Houston...
⏳ Extrayendo datos para Dallas...
⏳ Extrayendo datos para Austin...

Consolidando matrices climáticas...
¡Éxito! Dataset climático generado con 35064 registros horarios.
      timestamp_utc  texas_avg_temp  Houston_temp  Dallas_temp
0  2022-01-01T00:00       22.533333          23.2         20.4
1  2022-01-01T01:00       21.833333          22.8         19.8
2  2022-01-01T02:00       21.366667          23.2         18.8
3  2022-01-01T03:00       21.300000          23.6         18.8
4  2022-01-01T04:00       20.866667          23.2         18.0

[OK] Datos climáticos respaldados en: data/texas_weather_2022_2025_static.csv
